# Lily 1.5b v0.3 GGUF Conversion & Quantization — Google Colab
**Builds `llama.cpp` binaries, converts Lily-1.5b-v0.3 to GGUF format, and runs K-quantization**

Clones `llama.cpp`, patches system prompt configurations, converts model weights to `F16.gguf`, executes K-quantization (`Q4_K_M`, `Q5_K_M`, `Q8_0`), and uploads GGUF files to `abhinav0231/Lily-1.5b-v0.3-GGUF`.

## Cell 1 — Clone `llama.cpp` & Compile Binaries

In [ ]:
# ==============================================================================
# Cell 1 — Build llama.cpp Binary Tools via CMake
# ==============================================================================
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!mkdir build && cd build && cmake .. && make -j4
%cd ..
print("✅ llama.cpp binaries built successfully")

## Cell 2 — Download Base Model (`abhinav0231/Lily-1.5b-v0.3`)

In [ ]:
# ==============================================================================
# Cell 2 — Download Model Weights from Hugging Face Hub
# ==============================================================================
from huggingface_hub import snapshot_download

repo_id = "abhinav0231/Lily-1.5b-v0.3"
local_dir = "/content/Lily-1.5b-v0.3"

print(f"Downloading model snapshot: {repo_id} ...")
snapshot_download(repo_id=repo_id, local_dir=local_dir)
print(f"✅ Model downloaded to: {local_dir}")

## Cell 3 — Convert PyTorch Model to F16 GGUF

In [ ]:
# ==============================================================================
# Cell 3 — Execute convert_hf_to_gguf.py
# ==============================================================================
!python llama.cpp/convert_hf_to_gguf.py /content/Lily-1.5b-v0.3 --outtype f16 --outfile /content/Lily-1.5b-v0.3-F16.gguf
print("✅ F16 GGUF file generated")

## Cell 4 — Execute K-Quantization (`Q4_K_M`, `Q5_K_M`, `Q8_0`)

In [ ]:
# ==============================================================================
# Cell 4 — Quantize GGUF to 4-bit, 5-bit, and 8-bit Variants
# ==============================================================================
!llama.cpp/build/bin/llama-quantize /content/Lily-1.5b-v0.3-F16.gguf /content/Lily-1.5b-v0.3-Q4_K_M.gguf Q4_K_M
!llama.cpp/build/bin/llama-quantize /content/Lily-1.5b-v0.3-F16.gguf /content/Lily-1.5b-v0.3-Q5_K_M.gguf Q5_K_M
!llama.cpp/build/bin/llama-quantize /content/Lily-1.5b-v0.3-F16.gguf /content/Lily-1.5b-v0.3-Q8_0.gguf Q8_0
print("✅ Quantization complete")

## Cell 5 — Upload GGUF Package to Hugging Face Hub

In [ ]:
# ==============================================================================
# Cell 5 — Push Quantized GGUF Files to HF Hub Repository
# ==============================================================================
from huggingface_hub import HfApi
import os

target_repo = "abhinav0231/Lily-1.5b-v0.3-GGUF"
token = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN_HERE")

api = HfApi()
api.create_repo(target_repo, token=token, exist_ok=True)

gguf_files = [
    "/content/Lily-1.5b-v0.3-Q4_K_M.gguf",
    "/content/Lily-1.5b-v0.3-Q5_K_M.gguf",
    "/content/Lily-1.5b-v0.3-Q8_0.gguf",
]

for g_path in gguf_files:
    if os.path.exists(g_path):
        fname = os.path.basename(g_path)
        print(f"Uploading {fname} to {target_repo}...")
        api.upload_file(path_or_fileobj=g_path, path_in_repo=fname, repo_id=target_repo, token=token)

print(f"🎉 All GGUF files successfully published to: https://huggingface.co/{target_repo}")